# Stage 4 (v2) — Compound-shift-aware checkpoint selection, self-consistent rebuild

**Why this version exists**: the previous attempt tried to reproduce the original
Stage 1/2 pipeline's stylometric scaler exactly (refit-and-compare), which failed with
a real, large discrepancy (max abs diff = 10.97 standard-deviation-units) — meaning
whatever originally produced `Xs['train']` computes at least one feature differently
than this reimplementation (GPT-2 perplexity is the most likely culprit: unbounded,
version/truncation-sensitive, unlike the [0,1]-bounded lexical/POS features). Chasing
an exact match to code that was never shared is fragile and will keep breaking.

**The fix**: stop trying to match the original pipeline byte-for-byte. Recompute
stylometric features **fresh and consistently** for every split — train, val (original,
reddit), val_compound (new, reddit+dolly), testA, testB, testC — using one extractor and
one scaler fit once on train. Then retrain **both** the original-selection and
compound-selection Hybrid variants from scratch on this consistent feature set, so the
comparison is genuinely apples-to-apples regardless of what the original pipeline did
internally. This also means a real paired McNemar test is possible this time (both
variants' raw predictions come from the same session), which wasn't available before.

BERT embeddings are unaffected by this change (they operate on raw text, not stylometric
features) and are still reused from cache where available.

## 0. Force a clean rerun (only needed if you have run Stage 4 before)

In [ ]:
import glob, os

_CKPT_DIR_PRECHECK = '/kaggle/working/checkpoints'
stale = (glob.glob(f'{_CKPT_DIR_PRECHECK}/m4_hybrid_v2_compoundval_seed*.pkl') +
         glob.glob(f'{_CKPT_DIR_PRECHECK}/m4_hybrid_original_seed*.pkl') +
         glob.glob(f'{_CKPT_DIR_PRECHECK}/m4_style_train_raw_refit.pkl') +
         glob.glob(f'{_CKPT_DIR_PRECHECK}/m4_val_compound_style.pkl') +
         glob.glob(f'{_CKPT_DIR_PRECHECK}/m4_consistent_features.pkl'))
if stale:
    print(f'Removing {len(stale)} stale checkpoint(s) from a previous Stage-4 attempt:')
    for f in stale:
        print(' ', f)
        os.remove(f)
else:
    print('No previous Stage-4 checkpoints found -- nothing to clean.')


## 1. Setup + restore checkpoints from previous stages

In [ ]:
!pip install -q transformers accelerate scikit-learn pandas numpy nltk tqdm matplotlib shap statsmodels

import os, re, json, random, warnings, math, pickle, shutil, glob
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
from collections import Counter
from tqdm.auto import tqdm

import nltk
nltk.download('punkt', quiet=True); nltk.download('punkt_tab', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True); nltk.download('averaged_perceptron_tagger_eng', quiet=True)
nltk.download('stopwords', quiet=True)
from nltk import word_tokenize, sent_tokenize, pos_tag
from nltk.corpus import stopwords

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, GPT2LMHeadModel, get_cosine_schedule_with_warmup
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from statsmodels.stats.contingency_tables import mcnemar

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

CHECKPOINT_DIR = '/kaggle/working/checkpoints'
ARTIFACTS_DIR = '/kaggle/working/artifacts'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(ARTIFACTS_DIR, exist_ok=True)

restored = 0
for candidate in glob.glob('/kaggle/input/*/checkpoints') + glob.glob('/kaggle/input/*/*/checkpoints'):
    if os.path.isdir(candidate) and os.path.abspath(candidate) != os.path.abspath(CHECKPOINT_DIR):
        for item in os.listdir(candidate):
            dst = os.path.join(CHECKPOINT_DIR, item)
            if not os.path.exists(dst):
                src_path = os.path.join(candidate, item)
                (shutil.copytree if os.path.isdir(src_path) else shutil.copy2)(src_path, dst)
                restored += 1
        break
for candidate in glob.glob('/kaggle/input/*/artifacts') + glob.glob('/kaggle/input/*/*/artifacts'):
    if os.path.isdir(candidate) and os.path.abspath(candidate) != os.path.abspath(ARTIFACTS_DIR):
        for item in os.listdir(candidate):
            dst = os.path.join(ARTIFACTS_DIR, item)
            if not os.path.exists(dst):
                src_path = os.path.join(candidate, item)
                (shutil.copytree if os.path.isdir(src_path) else shutil.copy2)(src_path, dst)
                restored += 1
        break
print(f'Restored {restored} item(s) from a previous session (0 is normal if continuing the same session).')

def ckpt_path(name): return os.path.join(CHECKPOINT_DIR, name + '.pkl')
def ckpt_exists(name): return os.path.exists(ckpt_path(name))
def ckpt_save(name, obj):
    tmp = ckpt_path(name) + '.tmp'
    with open(tmp, 'wb') as f: pickle.dump(obj, f)
    os.replace(tmp, ckpt_path(name))
def ckpt_load(name):
    with open(ckpt_path(name), 'rb') as f: return pickle.load(f)

STOPWORDS = set(stopwords.words('english'))
FUNCTION_WORDS = ['the', 'of', 'and', 'to', 'in', 'is', 'that', 'it']
SEEDS = [42, 43, 44, 45, 46]


## 2. Load stage-2 splits (text + labels only — features recomputed fresh below)

In [ ]:
def find_artifact(*candidates):
    for c in candidates:
        if os.path.exists(c):
            return c
    raise FileNotFoundError(f'None of these paths exist: {candidates}')

stage2_path = find_artifact(
    '/kaggle/input/datasets/amanhunyawr/m4-hybrid-stage2/m4_stage2.pkl',  # your exact attached dataset path
    f'{ARTIFACTS_DIR}/m4_stage2.pkl', '/kaggle/working/m4_stage2.pkl',
    *glob.glob('/kaggle/input/*/m4_stage2.pkl'), *glob.glob('/kaggle/input/*/*/m4_stage2.pkl'))
print('Loading stage-2 artifact from:', stage2_path)
with open(stage2_path, 'rb') as f:
    stage2 = pickle.load(f)

splits = stage2['splits']   # only 'text'/'label' columns are used from here on -- Xs/Xcat/scaler from this pickle are NOT reused
df_train, df_val = splits['train'], splits['val']
y_train, y_val = df_train.label.values, df_val.label.values
test_names = ['testA', 'testB', 'testC']
y_test = {name: splits[name].label.values for name in test_names}
print('Loaded splits:', {k: len(v) for k, v in splits.items()})


## 3. Build `val_compound` — reddit domain + dolly generator, disjoint from every existing split

Unchanged from before — this part was never the problem and was verified against real
data (2,000 spare human docs, 3,000 dolly docs in reddit, zero overlap with any split).

In [ ]:
if not os.path.exists('/kaggle/working/M4'):
    !git clone -q https://github.com/mbzuai-nlp/M4.git /kaggle/working/M4
M4_ROOT = '/kaggle/working/M4/data'
MIN_WORDS = 30

def _wc(t): return len(t.split())
def clean(t): return re.sub(r'\s+', ' ', str(t)).strip()

def load_standard(path, domain, generator):
    human_rows, ai_rows = [], []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if not line: continue
            try: r = json.loads(line)
            except json.JSONDecodeError: continue
            ht, mt = clean(r.get('human_text', '')), clean(r.get('machine_text', ''))
            if _wc(ht) >= MIN_WORDS: human_rows.append({'text': ht, 'domain': domain, 'generator': 'human'})
            if _wc(mt) >= MIN_WORDS: ai_rows.append({'text': mt, 'domain': domain, 'generator': generator})
    return human_rows, ai_rows

reddit_human_raw, reddit_dolly_raw = [], []
for gen, fname in [('chatGPT','reddit_chatGPT.jsonl'), ('dolly','reddit_dolly.jsonl')]:
    path = os.path.join(M4_ROOT, fname)
    h, a = load_standard(path, 'reddit', gen)
    if gen == 'chatGPT': reddit_human_raw = h
    if gen == 'dolly': reddit_dolly_raw = a

df_reddit_human = pd.DataFrame(reddit_human_raw).drop_duplicates(subset=['text'])
df_reddit_dolly = pd.DataFrame(reddit_dolly_raw).drop_duplicates(subset=['text'])
df_reddit_human['label'] = 0
df_reddit_dolly['label'] = 1

VC_SEED = 43
N_VC = 300

used_val_human = set(df_val[df_val.label==0]['text'])
vc_human_pool = df_reddit_human[~df_reddit_human.text.isin(used_val_human)]
assert len(vc_human_pool) >= N_VC
vc_human = vc_human_pool.sample(n=N_VC, random_state=VC_SEED)
assert len(df_reddit_dolly) >= N_VC
vc_ai = df_reddit_dolly.sample(n=N_VC, random_state=VC_SEED)

df_val_compound = pd.concat([vc_human, vc_ai], ignore_index=True).sample(frac=1, random_state=VC_SEED).reset_index(drop=True)
y_val_compound = df_val_compound.label.values

all_existing_text = set()
for name, d in splits.items():
    all_existing_text |= set(d['text'])
overlap = set(df_val_compound['text']) & all_existing_text
assert len(overlap) == 0, f'LEAKAGE: {len(overlap)} val_compound texts already in an existing split'
print(f'val_compound: {len(df_val_compound)} samples, confirmed disjoint from every existing split.')


## 4. Consistent stylometric feature extraction — ONE extractor, ONE scaler, ALL six splits

This is the actual fix. Every split (train, val, val_compound, testA, testB, testC) goes
through the identical extractor fitted once on `df_train`, and the identical scaler fitted
once on the resulting train features. No cross-environment reproduction of a different
pipeline is attempted anywhere.

In [ ]:
def safe_div(a,b): return a/b if b else 0.0
def lexical_features(tokens):
    n = len(tokens)
    if n == 0: return {'hapax_ratio':0.,'yules_k':0.,'ttr':0.,'avg_word_len':0.}
    freqs = Counter(tokens); V = len(freqs)
    hapax = sum(1 for w,c in freqs.items() if c==1)
    freq_of_freq = Counter(freqs.values())
    sum_i2fi = sum((i**2)*fi for i,fi in freq_of_freq.items())
    return {'hapax_ratio':hapax/n,'yules_k':1e4*(sum_i2fi-n)/(n**2),'ttr':V/n,
            'avg_word_len':float(np.mean([len(w) for w in tokens]))}
def syntactic_features(sentences):
    lens = [len(word_tokenize(s)) for s in sentences] if sentences else [0]
    mean_len, var_len = float(np.mean(lens)), float(np.var(lens))
    burstiness = (var_len-mean_len)/(var_len+mean_len) if (var_len+mean_len)>0 else 0.
    full_text = ' '.join(sentences); n_chars = max(len(full_text),1)
    n_punct = sum(1 for c in full_text if c in '.,;:!?')
    return {'sent_len_variance':var_len,'burstiness':burstiness,'punct_density':n_punct/n_chars,'avg_sent_len':mean_len}
def pos_bigrams(tagged):
    tags = [t for _,t in tagged]; return list(zip(tags, tags[1:]))
def fit_pos_bigram_vocab(train_texts, top_k=36):
    counter = Counter()
    for text in tqdm(train_texts, desc='Fitting POS-bigram vocab (train only)'):
        counter.update(pos_bigrams(pos_tag(word_tokenize(text))))
    return [bg for bg,_ in counter.most_common(top_k)]
def grammatical_features(tagged, vocab):
    bigrams = pos_bigrams(tagged); total = len(bigrams); counts = Counter(bigrams)
    return {f'pos_{a}_{b}': (counts.get((a,b),0)/total if total>0 else 0.) for a,b in vocab}
def build_burrows_reference(train_human_texts, top_words):
    rates = {w: [] for w in top_words}
    for text in train_human_texts:
        tokens = [t.lower() for t in word_tokenize(text)]; n = max(len(tokens),1); freqs = Counter(tokens)
        for w in top_words: rates[w].append(freqs.get(w,0)/n)
    return ({w: float(np.mean(v)) for w,v in rates.items()}, {w: (float(np.std(v)) if np.std(v)>0 else 1.0) for w,v in rates.items()})
def burrows_delta(tokens, ref_mean, ref_std, top_words):
    n = max(len(tokens),1); freqs = Counter(tokens)
    diffs = [abs(((freqs.get(w,0)/n)-ref_mean.get(w,0.))/ref_std.get(w,1.)) for w in top_words]
    return float(np.mean(diffs)) if diffs else 0.
def function_word_ratios(tokens):
    n = max(len(tokens),1); freqs = Counter(t.lower() for t in tokens)
    return {f'func_{w}': freqs.get(w,0)/n for w in FUNCTION_WORDS}

class StylometricExtractor:
    def fit(self, train_texts, train_human_texts, n_bigrams=36, n_burrows_words=20):
        self.bigram_vocab = fit_pos_bigram_vocab(train_texts, n_bigrams)
        all_tok = [w.lower() for t in train_human_texts for w in word_tokenize(t)]
        self.burrows_words = [w for w,_ in Counter(all_tok).most_common(n_burrows_words) if w.isalpha()]
        self.ref_mean, self.ref_std = build_burrows_reference(train_human_texts, self.burrows_words)
        return self
    def transform(self, text, gpt2_ppl_fn=None):
        tokens, sentences = word_tokenize(text), sent_tokenize(text)
        tagged = pos_tag(tokens)
        feats = {}
        feats.update(lexical_features([t.lower() for t in tokens]))
        feats.update(syntactic_features(sentences))
        feats.update(grammatical_features(tagged, self.bigram_vocab))
        feats['burrows_delta'] = burrows_delta([t.lower() for t in tokens], self.ref_mean, self.ref_std, self.burrows_words)
        feats['gpt2_perplexity'] = gpt2_ppl_fn(text) if gpt2_ppl_fn else np.nan
        feats.update(function_word_ratios(tokens))
        return feats
    def transform_batch(self, texts, gpt2_ppl_fn=None, desc='Extracting'):
        rows = [self.transform(t, gpt2_ppl_fn) for t in tqdm(texts, desc=desc)]
        return pd.DataFrame(rows)

CAT_COLS = {
    'lexical': ['hapax_ratio','yules_k','ttr','avg_word_len'],
    'syntactic': ['sent_len_variance','burstiness','punct_density','avg_sent_len'],
}  # grammatical/authorship filled in after extractor.fit() below, since bigram_vocab/burrows_words are dynamic

_gpt2_tok = AutoTokenizer.from_pretrained('gpt2')
_gpt2_lm = GPT2LMHeadModel.from_pretrained('gpt2').to(DEVICE).eval()

@torch.no_grad()
def gpt2_perplexity(text, max_len=512):
    ids = _gpt2_tok(text, return_tensors='pt', truncation=True, max_length=max_len).input_ids.to(DEVICE)
    if ids.shape[1] < 2: return float('nan')
    loss = _gpt2_lm(ids, labels=ids).loss
    return float(torch.exp(loss).item())

extractor_ckpt = 'm4_v2_stylometric_extractor'
if ckpt_exists(extractor_ckpt):
    extractor = ckpt_load(extractor_ckpt)
else:
    extractor = StylometricExtractor().fit(df_train['text'].tolist(), df_train[df_train.label==0]['text'].tolist())
    ckpt_save(extractor_ckpt, extractor)

CAT_COLS['grammatical'] = [f'pos_{a}_{b}' for a,b in extractor.bigram_vocab]
CAT_COLS['authorship'] = ['burrows_delta', 'gpt2_perplexity'] + [f'func_{w}' for w in FUNCTION_WORDS]
feature_columns = CAT_COLS['lexical'] + CAT_COLS['syntactic'] + CAT_COLS['grammatical'] + CAT_COLS['authorship']
print('Total feature dimension:', len(feature_columns), '(expect 54)')
assert len(feature_columns) == 54


In [ ]:
vc_style_ckpt = 'm4_v2_val_compound_style'
if ckpt_exists(vc_style_ckpt):
    X_style_vc = ckpt_load(vc_style_ckpt)
else:
    X_style_vc = extractor.transform_batch(df_val_compound['text'].tolist(), gpt2_perplexity, 'val_compound style')
    ckpt_save(vc_style_ckpt, X_style_vc)

X_style = {'val_compound': X_style_vc}
all_splits = {'train': df_train, 'val': df_val, **{n: splits[n] for n in test_names}}
for name, d in all_splits.items():
    ckpt_name = f'm4_v2_style_{name}'
    if ckpt_exists(ckpt_name):
        X_style[name] = ckpt_load(ckpt_name)
    else:
        X_style[name] = extractor.transform_batch(d['text'].tolist(), gpt2_perplexity, f'style: {name}')
        ckpt_save(ckpt_name, X_style[name])

ppl_median = X_style['train']['gpt2_perplexity'].median()
for k in X_style:
    X_style[k]['gpt2_perplexity'] = X_style[k]['gpt2_perplexity'].fillna(ppl_median)

style_scaler = StandardScaler().fit(X_style['train'].values)
Xs = {k: style_scaler.transform(v[feature_columns].values) for k, v in X_style.items()}
col_index = {c: i for i, c in enumerate(feature_columns)}
def split_by_cat(arr): return {cat: arr[:, [col_index[c] for c in cols]] for cat, cols in CAT_COLS.items()}
Xcat = {k: split_by_cat(v) for k, v in Xs.items()}
category_dims = {k: len(v) for k, v in CAT_COLS.items()}

print('All stylometric feature ranges (should all be similar, standardized -- this is the actual fix):')
for name, arr in Xs.items():
    print(f'  {name:14s}: shape={arr.shape}, range=[{arr.min():.2f}, {arr.max():.2f}]')


## 5. Frozen BERT embeddings — reused from cache where available, val/val_compound computed fresh

In [ ]:
_bert_tok = AutoTokenizer.from_pretrained('distilbert-base-uncased')
_bert_model = AutoModel.from_pretrained('distilbert-base-uncased').to(DEVICE).eval()

@torch.no_grad()
def get_cls_embeddings(texts, batch_size=32, max_length=512, desc='BERT embeddings'):
    embs = []
    for i in tqdm(range(0, len(texts), batch_size), desc=desc):
        batch = texts[i:i+batch_size]
        enc = _bert_tok(batch, padding=True, truncation=True, max_length=max_length, return_tensors='pt').to(DEVICE)
        out = _bert_model(**enc).last_hidden_state[:, 0, :]
        embs.append(out.cpu().numpy())
    return np.vstack(embs)

def load_bert_npy(name):
    for path in [f'{ARTIFACTS_DIR}/{name}.npy', f'/kaggle/working/{name}.npy',
                 *glob.glob(f'/kaggle/input/*/{name}.npy'), *glob.glob(f'/kaggle/input/*/*/{name}.npy')]:
        if os.path.exists(path):
            return np.load(path)
    return None

Xb = {}
for name, d in all_splits.items():
    arr = load_bert_npy(f'm4_bert_{name}')
    if arr is None:
        print(f'm4_bert_{name}.npy not found -- computing.')
        arr = get_cls_embeddings(d['text'].tolist(), desc=f'BERT: {name}')
        np.save(f'{ARTIFACTS_DIR}/m4_bert_{name}.npy', arr)
    Xb[name] = arr

vc_bert_ckpt = 'm4_v2_val_compound_bert'
if ckpt_exists(vc_bert_ckpt):
    Xb['val_compound'] = ckpt_load(vc_bert_ckpt)
else:
    Xb['val_compound'] = get_cls_embeddings(df_val_compound['text'].tolist(), desc='BERT: val_compound')
    ckpt_save(vc_bert_ckpt, Xb['val_compound'])

print('BERT embedding shapes:', {k: v.shape for k, v in Xb.items()})


## 6. Attention-gated model + trainer

In [ ]:
class AttentionGatedHybridDetector(nn.Module):
    def __init__(self, category_dims, bert_dim=768, cat_hidden=32, ffn_out=64):
        super().__init__()
        self.category_names = list(category_dims.keys())
        self.category_encoders = nn.ModuleDict({
            name: nn.Sequential(nn.Linear(dim, cat_hidden), nn.ReLU(), nn.Dropout(0.2))
            for name, dim in category_dims.items()})
        n_cat = len(category_dims)
        self.gate = nn.Linear(cat_hidden * n_cat, n_cat)
        self.style_proj = nn.Sequential(nn.Linear(cat_hidden, ffn_out), nn.ReLU())
        self.classifier = nn.Sequential(
            nn.Linear(bert_dim + ffn_out, 256), nn.ReLU(), nn.Dropout(0.3), nn.Linear(256, 2))
    def forward(self, bert_emb, category_feats, return_gate=False):
        encoded = [self.category_encoders[name](category_feats[name]) for name in self.category_names]
        concat_encoded = torch.cat(encoded, dim=1)
        gate_weights = torch.softmax(self.gate(concat_encoded), dim=1)
        stacked = torch.stack(encoded, dim=1)
        weighted = (stacked * gate_weights.unsqueeze(-1)).sum(dim=1)
        style_repr = self.style_proj(weighted)
        logits = self.classifier(torch.cat([bert_emb, style_repr], dim=1))
        return (logits, gate_weights) if return_gate else logits

class CategoryDataset(Dataset):
    def __init__(self, bert_emb, cat_dict, labels):
        self.bert_emb = torch.tensor(bert_emb, dtype=torch.float32)
        self.cat_dict = {k: torch.tensor(v, dtype=torch.float32) for k, v in cat_dict.items()}
        self.labels = torch.tensor(labels, dtype=torch.long)
    def __len__(self): return len(self.labels)
    def __getitem__(self, i): return self.bert_emb[i], {k: v[i] for k, v in self.cat_dict.items()}, self.labels[i]

def collate_cat(batch):
    bert_embs = torch.stack([b[0] for b in batch])
    cat_names = batch[0][1].keys()
    cats = {name: torch.stack([b[1][name] for b in batch]) for name in cat_names}
    labels = torch.stack([b[2] for b in batch])
    return bert_embs, cats, labels

def make_cat_loader(bert_emb, cat_dict, labels, batch_size=64, shuffle=True):
    return DataLoader(CategoryDataset(bert_emb, cat_dict, labels), batch_size=batch_size, shuffle=shuffle, collate_fn=collate_cat)

def evaluate_gated(model, loader, threshold=0.5):
    model.eval(); all_labels, all_probs = [], []
    with torch.no_grad():
        for be, cats, lb in loader:
            be = be.to(DEVICE); cats = {k: v.to(DEVICE) for k, v in cats.items()}
            probs = torch.softmax(model(be, cats), dim=1)[:, 1]
            all_probs.extend(probs.cpu().numpy()); all_labels.extend(lb.numpy())
    all_labels, all_probs = np.array(all_labels), np.array(all_probs)
    return all_labels, (all_probs >= threshold).astype(int), all_probs

def calibrate_threshold(labels, probs, n_steps=199):
    best_t, best_f1 = 0.5, -1
    for t in np.linspace(0.01, 0.99, n_steps):
        f1 = f1_score(labels, (probs >= t).astype(int), zero_division=0)
        if f1 > best_f1: best_f1, best_t = f1, t
    return best_t, best_f1

def train_gated(train_loader, selection_loader, category_dims, seed, epochs=15, lr=1e-3, weight_decay=0.01, patience=5):
    torch.manual_seed(seed); np.random.seed(seed); random.seed(seed)
    model = AttentionGatedHybridDetector(category_dims).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    sched = get_cosine_schedule_with_warmup(opt, int(0.1*len(train_loader)*epochs), len(train_loader)*epochs)
    criterion = nn.CrossEntropyLoss()
    best_val_f1, best_state, epochs_no_improve = -1, None, 0
    for ep in range(1, epochs+1):
        model.train()
        for be, cats, lb in train_loader:
            be = be.to(DEVICE); cats = {k: v.to(DEVICE) for k, v in cats.items()}; lb = lb.to(DEVICE)
            opt.zero_grad()
            loss = criterion(model(be, cats), lb)
            loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); sched.step()
        sel_labels, sel_preds, _ = evaluate_gated(model, selection_loader)
        sel_f1 = f1_score(sel_labels, sel_preds, zero_division=0)
        if sel_f1 > best_val_f1:
            best_val_f1, best_state, epochs_no_improve = sel_f1, {k: v.clone() for k, v in model.state_dict().items()}, 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience: break
    model.load_state_dict(best_state)
    sel_labels, _, sel_probs = evaluate_gated(model, selection_loader)
    calibrated_t, _ = calibrate_threshold(sel_labels, sel_probs)
    return model, best_val_f1, calibrated_t

def full_metrics(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return {'accuracy': accuracy_score(y_true, y_pred), 'precision': precision_score(y_true, y_pred, zero_division=0),
            'recall': recall_score(y_true, y_pred, zero_division=0), 'f1': f1_score(y_true, y_pred, zero_division=0),
            'fpr': fp/(fp+tn) if (fp+tn) > 0 else 0.0}


## 7. Train both variants — 5 seeds each, same train data, same architecture

**Variant A (original)**: checkpoint selection on `val` (reddit, train generators).
**Variant B (compound)**: checkpoint selection on `val_compound` (reddit, dolly generator).

Both trained fresh in this session on the same consistently-computed features, so the
comparison below is genuinely apples-to-apples — unlike before, this can now use a real
paired McNemar test, since both variants' raw predictions come from this same run.

In [ ]:
train_loader = make_cat_loader(Xb['train'], Xcat['train'], y_train, batch_size=64, shuffle=True)
sel_loader_original = make_cat_loader(Xb['val'], Xcat['val'], y_val, batch_size=64, shuffle=False)
sel_loader_compound = make_cat_loader(Xb['val_compound'], Xcat['val_compound'], y_val_compound, batch_size=64, shuffle=False)
test_loaders = {name: make_cat_loader(Xb[name], Xcat[name], y_test[name], batch_size=128, shuffle=False) for name in test_names}

def run_variant(variant_name, selection_loader, ckpt_prefix):
    results = {name: [] for name in test_names}
    raw = {name: [] for name in test_names}
    gate_log = []
    for seed in SEEDS:
        ckpt_name = f'{ckpt_prefix}_seed{seed}'
        if ckpt_exists(ckpt_name):
            seed_result = ckpt_load(ckpt_name)
        else:
            model, best_sel_f1, calibrated_t = train_gated(train_loader, selection_loader, category_dims, seed=seed)
            seed_result = {'metrics': {}, 'raw': {}, 'gate': None, 'sel_f1': best_sel_f1, 'threshold': calibrated_t}
            for name in test_names:
                labels, preds, probs = evaluate_gated(model, test_loaders[name], threshold=calibrated_t)
                m = full_metrics(labels, preds); m['seed'] = seed; m['threshold'] = calibrated_t
                seed_result['metrics'][name] = m
                seed_result['raw'][name] = (labels, preds)
            with torch.no_grad():
                gates = []
                for be, cats, lb in test_loaders[test_names[0]]:
                    be = be.to(DEVICE); cats = {k: v.to(DEVICE) for k, v in cats.items()}
                    _, gw = model(be, cats, return_gate=True); gates.append(gw.cpu().numpy())
                seed_result['gate'] = dict(zip(category_dims.keys(), np.concatenate(gates).mean(axis=0).tolist()))
            ckpt_save(ckpt_name, seed_result)
            del model
            if torch.cuda.is_available(): torch.cuda.empty_cache()
        for name in test_names:
            results[name].append(seed_result['metrics'][name])
            raw[name].append(seed_result['raw'][name])
        gate_log.append(seed_result['gate'] | {'seed': seed})
        print(f'[{variant_name}] seed {seed}: sel_f1={seed_result["sel_f1"]:.4f}, threshold={seed_result["threshold"]:.3f}, ' +
              ', '.join(f'{n}_f1={seed_result["metrics"][n]["f1"]:.4f}' for n in test_names))
    return results, raw, pd.DataFrame(gate_log)

print('=== Variant A: original (reddit-val selection) ===')
results_original, raw_original, gates_original = run_variant('original', sel_loader_original, 'm4_hybrid_original')

print('\n=== Variant B: compound (reddit+dolly-val selection) ===')
results_compound, raw_compound, gates_compound = run_variant('compound', sel_loader_compound, 'm4_hybrid_v2_compoundval')


## 8. Compare — aggregate metrics, real paired McNemar, gate weights

In [ ]:
os.makedirs(ARTIFACTS_DIR, exist_ok=True)

comparison_rows = []
mcnemar_rows = []
for name in test_names:
    orig_df = pd.DataFrame(results_original[name])
    comp_df = pd.DataFrame(results_compound[name])
    comparison_rows.append({'test_set': name, 'variant': 'Original (reddit-val)',
                             'f1_mean': orig_df.f1.mean(), 'f1_std': orig_df.f1.std(),
                             'acc_mean': orig_df.accuracy.mean(), 'fpr_mean': orig_df.fpr.mean()})
    comparison_rows.append({'test_set': name, 'variant': 'Compound-val (reddit+dolly)',
                             'f1_mean': comp_df.f1.mean(), 'f1_std': comp_df.f1.std(),
                             'acc_mean': comp_df.accuracy.mean(), 'fpr_mean': comp_df.fpr.mean()})

    # Real paired McNemar this time -- both variants' seed-42 predictions come from this same run
    labels_orig, preds_orig = raw_original[name][0]
    labels_comp, preds_comp = raw_compound[name][0]
    assert (labels_orig == labels_comp).all(), 'label mismatch between variants -- should be identical'
    a_ok, b_ok = (preds_comp == labels_orig), (preds_orig == labels_orig)
    n01, n10 = int(np.sum(a_ok & ~b_ok)), int(np.sum(~a_ok & b_ok))
    result = mcnemar([[0, n01], [n10, 0]], exact=True)
    mcnemar_rows.append({'test_set': name, 'Compound_correct_Original_wrong': n01,
                          'Compound_wrong_Original_correct': n10, 'p_value': result.pvalue,
                          'significant_0.05': result.pvalue < 0.05,
                          'winner': 'Compound-val' if n01 > n10 else ('Original' if n10 > n01 else 'tie')})

comp_summary = pd.DataFrame(comparison_rows).round(4)
mcnemar_df = pd.DataFrame(mcnemar_rows).round(4)
print(comp_summary.to_string(index=False))
print()
print(mcnemar_df.to_string(index=False))

comp_summary.to_csv(f'{ARTIFACTS_DIR}/M4_v2_comparison.csv', index=False)
mcnemar_df.to_csv(f'{ARTIFACTS_DIR}/M4_v2_mcnemar.csv', index=False)
gates_original.to_csv(f'{ARTIFACTS_DIR}/M4_v2_gates_original.csv', index=False)
gates_compound.to_csv(f'{ARTIFACTS_DIR}/M4_v2_gates_compound.csv', index=False)

print()
print('=== Gate weights: ORIGINAL (reddit-val selection) ===')
print(gates_original.to_string(index=False))
print()
print('=== Gate weights: COMPOUND (reddit+dolly-val selection) ===')
print(gates_compound.to_string(index=False))
print()
print('If compound-val selection produces higher, more consistent Grammatical weights AND/OR')
print('significantly better testC F1 in the McNemar table above, the fix shows a real effect.')
print('If gate weights and testC performance look similar between variants, checkpoint-selection')
print('signal was not the bottleneck -- the compound-shift failure runs deeper than this.')
